In [2]:
%pip install anthropic python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     --- ------------------------------------ 10.2/109.4 kB ? eta -:--:--
     ---------- ---------------------------- 30.7/109.4 kB 1.3 MB/s eta 0:00:01
     -------------------- ---------------- 61.4/109.4 kB 544.7 kB/s eta 0:00:01
     ------------------------------- ----- 92.2/109.4 kB 655.4 kB/s eta 0:00:01
     ------------------------------------ 109.4/109.4 kB 527.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/956.9 kB ? eta -:--:--
   --- ------------------------------------ 92.2/956.9 kB 2.6 MB/s eta 0:00:01
   ---- ----------------------------------- 112.6/956.9 kB 2.2 MB/s eta 0:00:01
   --------- ------------------------------ 225.3/956.9 kB 1.5 MB/s eta 0:00:01
   --------------- ------------------------ 368.6/956.9 kB 2.1 MB/s eta 0:00:01
   ----------------- ---------------------- 419.8/956.9 kB 1.9 MB/s eta 0:


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\umaha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
# Load env variables from .env file
import os
from pathlib import Path

from dotenv import load_dotenv

workspace_root = Path(r"C:/AI-Training/Anthropic-learning/My-Anthropic-Training")
dotenv_path = workspace_root / ".env"
load_dotenv(dotenv_path, override=True)
print("API key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("dotenv path:", dotenv_path)

API key loaded: True
dotenv path: C:\AI-Training\Anthropic-learning\My-Anthropic-Training\.env


In [2]:
# create an API client
import os
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

workspace_root = Path(r"C:/AI-Training/Anthropic-learning/My-Anthropic-Training")
dotenv_path = workspace_root / ".env"
load_dotenv(dotenv_path, override=True)

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is not set. Add it to your .env file.")
if len(api_key) < 20:
    raise ValueError("ANTHROPIC_API_KEY looks too short to be valid.")

client = Anthropic(api_key=api_key)
model = "claude-sonnet-4-5"
print("Client ready")

Client ready


In [3]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[{"role": "user", "content": "What is Quantum Computing? Answer in one sentence."}],
)
print(message.content[0].text)

Quantum computing is a type of computing that uses quantum mechanical phenomena like superposition and entanglement to perform calculations that would be impossible or impractical for classical computers.


In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text


In [5]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
answer = chat(messages)

answer

'Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that are strictly either 0 or 1.'

In [6]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads .env and loads ANTHROPIC_API_KEY into the environment

client = Anthropic()  # automatically picks up ANTHROPIC_API_KEY from the environment

response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": (
                "Extract this contract: Nimbus Cloud Services, contract "
                "NCS-2025-0142, ends Sept 25 2026, auto-renews unless "
                "cancelled 30 days prior, $63,000/year."
            ),
        }
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "vendor": {"type": "string"},
                    "end_date": {"type": "string"},
                    "auto_renews": {"type": "boolean"},
                    "notice_days": {"type": "integer"},
                    "annual_value": {"type": "number"},
                },
                "required": [
                    "vendor", "end_date", "auto_renews",
                    "notice_days", "annual_value",
                ],
                "additionalProperties": False,
            },
        }
    },
)
print(next(b.text for b in response.content if b.type == "text"))

{"vendor":"Nimbus Cloud Services","end_date":"2026-09-25","auto_renews":true,"notice_days":30,"annual_value":63000}


In [ ]:
from anthropic import Anthropic

client = Anthropic()

contracts = [
    "Nimbus Cloud Services, ends Sept 25 2026, auto-renews unless cancelled 30 days prior.",
    "BrightPath Security Solutions, ends Oct 10 2026, auto-renews unless cancelled 45 days prior.",
    "Vertex Networking Group, ends Nov 5 2026, auto-renews unless cancelled 60 days prior.",
    "Alderwood Office Supplies, ends Dec 15 2026, auto-renews unless cancelled 60 days prior.",
    "Fixed-Term Consulting LLC, ends Sept 20 2026, fixed term, does not auto-renew.",
]

for contract in contracts:
    response = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": (
                "Classify this contract's renewal risk as HIGH, MEDIUM, "
                f"or LOW. Assume today is August 15, 2026.\n\n{contract}"
            ),
        }],
    )
    text = next((b.text for b in response.content if b.type == "text"), "No text response")
    print(f"--- {contract[:40]}...\n{text}\n")

--- Nimbus Cloud Services, ends Sept 25 2026...
# Renewal Risk Classification: **HIGH**

## Key Facts
- **Contract end date:** September 25, 2026
- **Today's date:** August 15, 2026
- **Cancellation deadline:** August 26, 2026 (30 days prior to end date)
- **Days remaining to cancel:** ~11 days

## Why This Is HIGH Risk

| Factor | Assessment |
|---|---|
| Time to deadline | Only **11 days** until the cancellation window closes |
| Action required | Auto-renewal means **silence = renewal** — no action defaults to a new term |
| Reversibility | Once Aug 26 passes, likely locked in for another full term |
| Buffer for review/decision | Minimal — barely enough time for internal review, budget

--- BrightPath Security Solutions, ends Oct ...
**Renewal Risk Classification: HIGH**

**Key Details:**
- **Contract End Date:** October 10, 2026
- **Notice Deadline:** Cancellation must occur 45 days prior = **August 26, 2026**
- **Today's Date:** August 15, 2026
- **Time Remaining to Decide/Act:**

StopIteration: 

In [10]:
def grade(result: dict, expected_risk: str) -> str:
    """Exact-match grader: no interpretation, just comparison."""
    actual = result.get("risk", "").strip().upper()
    expected = expected_risk.strip().upper()
    return "PASS" if actual == expected else f"FAIL (got {actual}, expected {expected})"

# Example usage against one of your 5 contracts:
sample_result = {"vendor": "Vertex Networking Group", "risk": "MEDIUM"}
print(grade(sample_result, expected_risk="MEDIUM"))

PASS


In [11]:
from anthropic import Anthropic

client = Anthropic()

rubric = """
PASS if: the risk label matches what you'd calculate by hand from the
end date, auto-renew status, and notice period, using August 1, 2026
as today.
FAIL if: the label is wrong, or no label is given.
"""

contract = "Vertex Networking Group, ends Nov 5 2026, auto-renews unless cancelled 60 days prior."
claimed_output = "MEDIUM"

response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": (
            f"Rubric:\n{rubric}\n\nContract: {contract}\n"
            f"Assume today is August 1, 2026.\n"
            f"Submitted answer: {claimed_output}\n\n"
            "Grade this PASS or FAIL against the rubric. Show your "
            "date-math reasoning first, then give the verdict."
        ),
    }],
)
print(next(b.text for b in response.content if b.type == "text"))

## Date-Math Reasoning

**Contract end date:** November 5, 2026
**Notice period required:** 60 days prior to end date (to cancel auto-renewal)
**Notice deadline:** November 5, 2026 − 60 days = **September 6, 2026**

**Today's date:** August 1, 2026

**Days remaining until notice deadline:**
- August 1 → August 31 = 30 days
- August 31 → September 6 =
